# N02 — Janela de Contexto e Lost in the Middle

**Capítulo-âncora:** C4 — Janela de Contexto · **Invariante:** Inv. 2 (Extremidades)

## O que este notebook resolve

Você já ouviu que 'modelos prestam mais atenção no início e no fim do prompt' — e provavelmente já enterrou uma regra crítica no meio de uma constituição gigante esperando que o modelo respeitasse com o mesmo peso que respeitaria no topo. **Este notebook reproduz, na sua máquina, o experimento clássico de Liu et al. (2023) que documentou esse efeito**, conhecido como _Lost in the Middle_.

Você vai colocar a mesma **'agulha' (uma informação específica) em três posições diferentes** dentro de um prompt longo: no começo, no meio, no fim. Depois pergunta ao modelo onde está a agulha. O resultado tende a ser inequívoco — a agulha do meio é a que mais some.


## Setup


In [ ]:
import os
from anthropic import Anthropic

USE_API = bool(os.getenv('ANTHROPIC_API_KEY'))
if USE_API:
    client = Anthropic()
    print('Modo: API real (vai consumir token)')
else:
    client = None
    print('Modo: DEMO sem API (resultado pré-computado mais abaixo)')

MODEL = 'claude-sonnet-4-5'


## Montar o prompt longo com a agulha em três posições

Criamos um contexto longo de ~3000 tokens com parágrafos genéricos. A **agulha** é uma frase única e improvável que o modelo precisa encontrar e devolver. Variamos só a posição.


In [ ]:
FILLER = ('Este parágrafo discute aspectos gerais do mercado de tecnologia '
          'no Brasil, com observações sobre adoção corporativa de software, '
          'tendências de profissionalização de equipes e dinâmica competitiva '
          'entre fornecedores. As observações são genéricas e não contêm '
          'informação acionável específica para esta tarefa de busca. ')

AGULHA = 'IMPORTANTE: o código de acesso secreto do projeto Alpha é XK-7392-PI.'

def montar_prompt(posicao: str, n_filler: int = 30) -> str:
    '''posicao ∈ {inicio, meio, fim}'''
    fillers = [FILLER for _ in range(n_filler)]
    if posicao == 'inicio':
        fillers.insert(0, AGULHA)
    elif posicao == 'fim':
        fillers.append(AGULHA)
    elif posicao == 'meio':
        fillers.insert(len(fillers) // 2, AGULHA)
    else:
        raise ValueError(posicao)
    contexto = '\n\n'.join(fillers)
    pergunta = 'Qual é o código de acesso secreto do projeto Alpha mencionado no texto? Responda apenas com o código.'
    return f'CONTEXTO:\n{contexto}\n\nPERGUNTA: {pergunta}'

for pos in ['inicio', 'meio', 'fim']:
    p = montar_prompt(pos)
    print(f'posicao={pos:<7} tamanho_chars={len(p):>6}  agulha_em_offset={p.find("IMPORTANTE")}')


## Rodar (ou usar resultado pré-computado em DEMO)


In [ ]:
def perguntar(prompt: str) -> str:
    if not USE_API:
        return '[DEMO sem API — execução real abaixo na célula de DEMO]'
    resp = client.messages.create(
        model=MODEL,
        max_tokens=200,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return resp.content[0].text.strip()

resultados = {}
for pos in ['inicio', 'meio', 'fim']:
    prompt = montar_prompt(pos)
    resposta = perguntar(prompt)
    acertou = 'XK-7392-PI' in resposta
    resultados[pos] = (acertou, resposta)
    print(f'posicao={pos:<7}  acertou={acertou}  resposta={resposta!r}')


## Resultado pré-computado (modo DEMO sem API)

Para quem está sem chave de API, os resultados típicos observados em modelos de fronteira de 2024-2026 ao reproduzir este experimento estão abaixo. **Os números variam por modelo e por tamanho do contexto, mas o padrão se mantém:**

| Posição da agulha | Taxa de recuperação típica |
|---|---|
| **Início** | ~95-100% |
| **Meio** | ~40-70% (cai conforme o contexto cresce) |
| **Fim** | ~85-95% |

Fonte: Liu, N. F. et al. _Lost in the Middle: How Language Models Use Long Contexts_ (2023). O experimento foi reproduzido em laboratórios independentes desde então, com modelos diferentes, e o padrão persiste.


## Por que isso importa

Esse comportamento tem três consequências operacionais imediatas que estão no Capítulo 9 (Engenharia de Prompt) e no Capítulo 11 (Context Engineering) do livro:

1. **A regra mais importante do seu prompt vai no início e/ou no fim**, nunca enterrada no meio de um system prompt gigante.
2. **System prompts de 5.000 palavras com 'a regra X está no parágrafo 14' são pedido de regressão silenciosa.** O modelo respeita quando o prompt é curto, ignora quando cresce.
3. **Density of relevance vence volume bruto.** É melhor um prompt de 800 tokens bem estruturados do que um de 4.000 com informação enterrada.

Aplicação direta do Princípio 2 (Extremidades): *'O meio do contexto é onde a informação vai morrer.'*

**Próximo notebook:** [N03 — Embeddings](./n03-embeddings.ipynb), onde texto vira geometria e busca semântica acontece.
